# 01 — Train and track experiments (MLflow)

Data-scientist workflow on OpenShift AI:

1. Train short YOLO runs with different hyperparameters
2. Log params, metrics, dataset lineage, and weights to **MLflow** (experiments only)
3. Do **not** register every run — selection happens in notebook 02

Set `MLFLOW_TRACKING_URI` (see `docs/env.md`). Defaults favor a live demo (`DEMO_EPOCHS=3`).

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "images" / "data.yaml").exists() and (REPO_ROOT.parent / "images" / "data.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import mlflow
from ultralytics import YOLO

from src.mlflow_utils import configure_tracking, log_dataset_lineage, log_yolo_metrics

DATA_YAML = "images/data.yaml"
EXPERIMENT = configure_tracking()
print("cwd:", Path.cwd())
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT)

## Experiment matrix

Run at least two configurations so the MLflow UI has something to compare.
Override via env: `DEMO_EPOCHS`, `DEMO_IMGSZ`, `DEMO_MODEL`.

In [ ]:
EPOCHS = int(os.environ.get("DEMO_EPOCHS", "3"))
IMGSZ = int(os.environ.get("DEMO_IMGSZ", "640"))
BASE_MODEL = os.environ.get("DEMO_MODEL", "yolov8n.pt")

RUN_CONFIGS = [
    {"run_name": "exp-lr-default", "epochs": EPOCHS, "imgsz": IMGSZ, "lr0": 0.01, "batch": 16},
    {"run_name": "exp-lr-low", "epochs": EPOCHS, "imgsz": IMGSZ, "lr0": 0.005, "batch": 16},
    {"run_name": "exp-imgsz-512", "epochs": EPOCHS, "imgsz": 512, "lr0": 0.01, "batch": 16},
]

# For a quicker single-run smoke test, uncomment:
# RUN_CONFIGS = RUN_CONFIGS[:1]

RUN_CONFIGS

In [ ]:
completed = []

for cfg in RUN_CONFIGS:
    run_name = cfg["run_name"]
    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        mlflow.log_params({
            "model": BASE_MODEL,
            "data": DATA_YAML,
            "epochs": cfg["epochs"],
            "imgsz": cfg["imgsz"],
            "lr0": cfg["lr0"],
            "batch": cfg["batch"],
            "run_name": run_name,
            "project": "runs-openshift",
        })
        log_dataset_lineage(DATA_YAML)

        model = YOLO(BASE_MODEL)
        results = model.train(
            data=DATA_YAML,
            epochs=cfg["epochs"],
            imgsz=cfg["imgsz"],
            lr0=cfg["lr0"],
            batch=cfg["batch"],
            project="runs-openshift",
            name=run_name,
            exist_ok=True,
        )

        metrics = getattr(results, "results_dict", None)
        if metrics is None and hasattr(model, "trainer") and model.trainer is not None:
            validator = getattr(model.trainer, "validator", None)
            metrics = getattr(validator, "metrics", None)
        recorded = {}
        if metrics is not None:
            recorded = log_yolo_metrics(metrics)

        weights_dir = Path("runs-openshift") / run_name / "weights"
        best_pt = weights_dir / "best.pt"
        if best_pt.is_file():
            mlflow.log_artifact(str(best_pt), artifact_path="weights")
            mlflow.set_tag("weights_local_path", str(best_pt))

        completed.append({"run_id": run_id, "run_name": run_name, "metrics": recorded, "best_pt": str(best_pt)})
        print(f"Finished {run_name} run_id={run_id} metrics={recorded}")

completed

Next: open **`02_evaluate_select.ipynb`** to compare runs in MLflow and pick a single candidate for registration.